In [69]:
import pandas as pd
import numpy as np

In [70]:
bm_raw = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bm_raw_new.csv")
bss_pca = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/bss_pca_raw.csv")
pca_sellers = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Consumption till 13th/PCA for Sellers.csv")
classification = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/classification_mapping.xlsx")
Brand_mapping = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Brand_mapping.xlsx")
search_roin = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/1.Jan'26/14.01/Search/Self serve dashboard.csv")

In [71]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [72]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [73]:
# bss_pca.columns

In [74]:
#  pca_sellers.columns

In [75]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [76]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

In [77]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [78]:
bm_raw['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
MP,452982
Alpha,42970
IC,2047


In [79]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [80]:
bm_raw['FCFF Alpha/MP'].value_counts()

,count
FCFF Alpha/MP,
MP,454825
Alpha,36818
HL,6356


In [81]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [82]:
# print(bm_raw['New Supercat'].value_counts().to_string())


In [83]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [84]:
bm_raw['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,497710
False,289


In [85]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [86]:
print(bm_raw['New BU'].isnull().sum())

289


In [87]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [88]:
bm_raw['Spends'].sum()

np.float64(1159516729.84)

In [89]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


In [90]:
len(bm_raw[bm_raw['Brand'] == "0"])


250

In [91]:
# bm_raw['Brand'].value_counts(normalize=True)

In [92]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR', 'New Alpha/MP', 'FCFF Alpha/MP', 'New Supercat',
       'SC match FCFF', 'New BU', 'Spends', 'lookup_key', 'Brand'],
      dtype='object')

# BSS PCA

In [93]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [94]:
bss_pca = bss_pca[bss_pca['spend']>5].copy()

In [95]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


In [96]:
bss_pca['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
Alpha,23472
MP,8969


In [97]:
class_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])


In [98]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

In [99]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [100]:
bss_pca['SC match FCFF'].value_counts()

,count
SC match FCFF,
True,31973
False,468


In [101]:
bss_pca[bss_pca['SC match FCFF'] == False]['New Supercat'].value_counts()

,count
New Supercat,
Not Assigned,468


In [102]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [103]:
print(bss_pca['New BU'].isnull().sum())

468


In [104]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

In [105]:
len(bss_pca[bss_pca['Brand'] == "0"])

393

In [106]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [144]:
#  bss_pca['FCFF Alpha/MP'].value_counts()

In [108]:
bss_pca['BUxBrand'] = bss_pca['New BU'].astype(str) + bss_pca['Brands'].astype(str)
lookup_map = dict(zip(bss_pca['BUxBrand'], bss_pca['New Supercat']))

bss_pca['Check'] = bss_pca['BUxBrand'].map(lookup_map)

In [109]:
# bss_pca.info()

In [110]:

len(bss_pca[bss_pca['SC match FCFF'] == False])


468

In [111]:
len(bss_pca[bss_pca['Supercategory'] == "Not Assigned"])

517

In [112]:
bss_pca['New Supercat'] = np.where(bss_pca['Check'] == "Not Assigned", "Not Assigned",np.where(bss_pca['Check'] == "PetCare", "PetCare",bss_pca["Check"]))

In [113]:
bss_pca['New BU'].value_counts()

,count
New BU,
BGM,14160
Lifestyle,4589
Electronics,4407
Grocery,3792
Home,3184
Large,1097
Furniture,449
Mobile,295


In [114]:
# bss_pca['New Supercat'].value_counts()

In [115]:
print(bss_pca['New BU'].isnull().sum())

468


In [116]:
print(round(bss_pca[bss_pca['SC match FCFF'] == False ]['spend'].sum(),1))

230599.0


# Search ROINS

In [117]:
search_roin.head()

,PURCHASE_ORDER_ID,PURCHASE_ORDER_NAME,AMOUNT,START_DATE,END_DATE,PAYMENT_CYCLE,ACCOUNT_ID,ACCOUNT_NAME,BUSINESS_ACCOUNT_ID,BUSINESS_ACCOUNT_NAME,CREATED_AT,UPDATED_AT
0,00NP1Q1N5F9Z,ECO1020326006801,8000000.0,2026-01-12,NaN,30,YWVXLNIMY3JS,BoultAudio - RO,ZTTKSHCHAAK9,EXOTIC MILE PRIVATE LIMITED,2026-01-12T12:32:42.000Z,2026-01-12T12:32:42.000Z
1,03CNFWSQ6TI1,R266/OT/26/000068/00,90000.0,2026-01-02,2026-01-31,60,TPJDJFPSBNV3,Colgate-Minutes,AXTWDULDJS8Z,Colgate Palmolive India Limited,2026-01-02T06:04:58.000Z,2026-01-02T06:04:58.000Z
2,03V43PQ10X2Y,EmamiAgroHealthy&Tasty/Spices/Jan'26,200000.0,2026-01-01,NaN,30,10N825F1XMU4,Emami Mantra H&T_Grocery PO,8ZHLXE1UH284,Emami Agrotech Ltd,2026-01-01T04:55:01.000Z,2026-01-01T04:55:01.000Z
3,05Y902DHG1JY,AGLPO129645,1000000.0,2026-01-10,NaN,30,WDAPRI0KKX,Eveready_PO,org-WDAPRI0KKX,EVEREADY INDUSTRIES INDIA LTD,2026-01-10T10:04:25.000Z,2026-01-10T10:04:25.000Z
4,0720VUH9VLWT,R167/OT/26/000113/01,105000.0,2026-01-01,2026-01-31,30,TK9W1XK3954O,FKMin_Durex,UKLJEXGRY7HR,Reckitt Benckiser (India) Private Limited,2026-01-12T11:36:45.000Z,2026-01-12T11:36:45.000Z


- Start and End of Current Month

In [129]:
SOM = "2026-01-01"
EOM = "2026-01-31"

In [130]:
search_ro = search_roin[(search_roin['START_DATE'] >= SOM ) & (search_roin['START_DATE'] <= EOM) ].copy()

In [131]:
# search_ro.head()

In [132]:
bm_raw_ro = bm_raw[bm_raw['SC match FCFF'] == True].copy()

In [133]:
bm_raw_ro['total_cost_x'].sum()

np.float64(1159380542.6100001)

In [138]:
bss_pca_ro = bss_pca[bss_pca['SC match FCFF'] == True].copy()

In [142]:
bss_pca_ro['spend'].sum()

np.float64(98359522.83999997)

- Alpha/MP

In [159]:
# alpha_mp_map_bm_raw = dict(zip(bm_raw_ro['advertiser_id'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['FCFF Alpha/MP'].to_dict()
# alpha_mp_map_bss_pca = dict(zip(bss_pca_ro['Ad Account ID'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['FCFF Alpha/MP'].to_dict()

search_ro['Alpha_MP'] = search_ro['ACCOUNT_ID'].map(alpha_mp_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(alpha_mp_map_bss_pca))

In [161]:
print(round(search_ro.groupby('Alpha_MP')['AMOUNT'].sum()))

Alpha_MP
Alpha    657527266.0
HL        35487747.0
MP       186037882.0
Name: AMOUNT, dtype: float64


- BU

In [162]:
bu_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New BU'].to_dict()
bu_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New BU'].to_dict()

search_ro['New BU'] = search_ro['ACCOUNT_ID'].map(bu_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(bu_map_bss_pca))

In [163]:
search_ro['New BU'].value_counts()

,count
New BU,
BGM,338
Grocery,119
Large,115
Lifestyle,103
Electronics,65
Home,22
Furniture,8
Mobile,4


- Brand


In [165]:
brand_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['brand'].to_dict()
brand_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['Brands'].to_dict()

search_ro['New Brand'] = search_ro['ACCOUNT_ID'].map(brand_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(brand_map_bss_pca))

- Super Category

In [166]:
sc_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New Supercat'].to_dict()
sc_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New Supercat'].to_dict()

search_ro['New Supercategory'] = search_ro['ACCOUNT_ID'].map(sc_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(sc_map_bss_pca))

In [167]:
search_ro['New Supercategory'].value_counts()

,count
New Supercategory,
Grocery,119
Grooming,101
FoodAndNutrition,86
HouseHoldSupplies,36
HealthCare,35
SeasonalEA,23
LuggageAndTravelAccessories,22
BabyCare,21
MakeupFragrances,21
